# LeetCode 888 — Fair Candy Swap bằng Python

**Link đề:** https://leetcode.com/problems/fair-candy-swap/
**Độ khó:** Easy · **Chủ đề:** Array, Hash Table, Binary Search, Sorting

```python
class Solution:
    def fairCandySwap(self, aliceSizes: list[int], bobSizes: list[int]) -> list[int]:
        ...
```

## Mục tiêu học
1. Biến câu chuyện "đổi hộp kẹo cho công bằng" thành **một phương trình** và giải ra `y = x + delta`.
2. Thấy rõ nút thắt của cách thử mọi cặp `O(n·m)` và cách **hash set** đưa về `O(n + m)` — họ hàng gần của Two Sum.
3. Thêm hai lời giải khác: **sắp xếp + hai con trỏ** và **sắp xếp + tìm kiếm nhị phân** (`bisect`).
4. Trace, chứng minh, phân tích Time/Space, so sánh 4 cách và kiểm thử bằng "bộ kiểm tra đáp án" (vì đề cho phép nhiều đáp án).
5. Luyện 6 bài tập từ tính tay tới mini challenge.

## 1. Đề bài và ví dụ

Alice có các hộp kẹo `aliceSizes`, Bob có `bobSizes` (`aliceSizes[i]` = số kẹo trong hộp `i`). Tổng kẹo của hai người **khác nhau**.

Hai người muốn **đổi đúng một hộp mỗi người** để sau khi đổi, **tổng kẹo bằng nhau**.

Trả về `[x, y]`: `x` = số kẹo của hộp Alice đưa đi, `y` = số kẹo của hộp Bob đưa đi. Có nhiều đáp án thì trả **bất kỳ**. Đề **đảm bảo** luôn có ít nhất một đáp án.

| aliceSizes | bobSizes | Output |
|---|---|---|
| `[1,1]` | `[2,2]` | `[1,2]` |
| `[1,2]` | `[2,3]` | `[1,2]` |
| `[2]` | `[1,3]` | `[2,3]` |

**Constraints:** `1 <= len <= 10^4`; `1 <= size <= 10^5`; tổng hai người khác nhau; luôn có đáp án.

### 1.1. Kiểm tra tay ví dụ 3

Alice `[2]` (tổng 2), Bob `[1,3]` (tổng 4). Đổi hộp `2` của Alice lấy hộp `3` của Bob:
- Alice: `2 - 2 + 3 = 3`
- Bob: `4 - 3 + 2 = 3` ✔ bằng nhau.

Thử đổi `2` lấy `1`: Alice `1`, Bob `5` ✘.

## 2. Nền tảng: biến bài toán thành phương trình

Gọi `SA = sum(aliceSizes)`, `SB = sum(bobSizes)`. Alice đưa hộp `x`, nhận hộp `y`:

```text
Alice sau đổi:  SA - x + y
Bob sau đổi:    SB - y + x
Cần bằng nhau:  SA - x + y = SB - y + x
             => 2y - 2x = SB - SA
             =>  y = x + (SB - SA) / 2
```

Đặt `delta = (SB - SA) / 2`. Vậy: **với mỗi hộp `x` của Alice, hộp Bob cần có đúng `y = x + delta` kẹo.**

Ví dụ 3: `SA = 2`, `SB = 4`, `delta = 1` → `x = 2` cần `y = 3` ✔ (Bob có 3).

In [1]:
alice, bob = [2], [1, 3]
SA, SB = sum(alice), sum(bob)
delta = (SB - SA) // 2
print("SA =", SA, "| SB =", SB, "| delta =", delta)
for x in alice:
    print(f"x = {x} -> cần y = x + delta = {x + delta}")

SA = 2 | SB = 4 | delta = 1
x = 2 -> cần y = x + delta = 3


### 2.1. Vì sao dùng `//` mà không sợ sai?

Vì đề đảm bảo **có** đáp án, `SB - SA = 2(y - x)` luôn là **số chẵn** → `(SB - SA) // 2` chia hết, không mất phần lẻ.

Chú ý `//` với số âm làm tròn **xuống** (`-3 // 2 = -2`), nhưng khi số bị chia chẵn thì kết quả chính xác (`-4 // 2 = -2`). Dùng `/` sẽ ra `float` (`1.0`), so sánh `1.0 == 1` vẫn `True` trong Python nhưng nên giữ số nguyên cho sạch.

In [2]:
print("4 // 2 =", 4 // 2, "| -4 // 2 =", -4 // 2)
print("-3 // 2 =", -3 // 2, "(làm tròn xuống, chỉ xảy ra nếu hiệu lẻ - không có trong đề)")
print("2 / 2 =", 2 / 2, type(2 / 2))

4 // 2 = 2 | -4 // 2 = -2
-3 // 2 = -2 (làm tròn xuống, chỉ xảy ra nếu hiệu lẻ - không có trong đề)
2 / 2 = 1.0 <class 'float'>


### 2.2. `sum()` và `set` — hai công cụ cần dùng

- `sum(list)` cộng mọi phần tử, `O(n)`.
- `set(list)` tạo **tập hợp**; kiểm tra `v in s` mất `O(1)` **trung bình** (nhờ băm — hashing), trong khi `v in list` mất `O(n)`.

In [3]:
import time
data = list(range(200_000))
s = set(data)
t0 = time.perf_counter(); _ = 199_999 in data; t1 = time.perf_counter()
_ = 199_999 in s; t2 = time.perf_counter()
print(f"in list: {t1 - t0:.6f}s | in set: {t2 - t1:.6f}s  (set nhanh hơn rất nhiều)")

in list: 0.001233s | in set: 0.000032s  (set nhanh hơn rất nhiều)


## 3. Cách 1 — Brute force: thử mọi cặp hộp

Chưa cần phương trình: thử mọi `x` của Alice và mọi `y` của Bob, kiểm tra trực tiếp `SA - x + y == SB - y + x`.

In [4]:
def fair_swap_bruteforce(alice: list[int], bob: list[int]) -> list[int]:
    SA, SB = sum(alice), sum(bob)
    for x in alice:
        for y in bob:
            if SA - x + y == SB - y + x:
                return [x, y]
    return []

print(fair_swap_bruteforce([1, 1], [2, 2]))
print(fair_swap_bruteforce([1, 2], [2, 3]))
print(fair_swap_bruteforce([2], [1, 3]))

[1, 2]
[1, 2]
[2, 3]


### 3.1. Nút thắt

Hai vòng lồng nhau → `O(n · m)`. Với `n = m = 10^4` là `10^8` cặp — quá chậm với Python (vài chục giây).

Nhìn lại: với **mỗi `x`**, ta đang duyệt toàn bộ Bob chỉ để hỏi một câu: *"Bob có hộp nào đúng bằng `x + delta` không?"*

Câu hỏi "có tồn tại giá trị `v` không?" → trả lời trong `O(1)` trung bình bằng **set**. Đây là cùng ý tưởng với Two Sum (`needed = target - x`).

## 4. Cách 2 — Lời giải chính: phương trình + hash set

1. Tính `delta = (SB - SA) // 2`.
2. `bob_set = set(bob)`.
3. Với mỗi `x` trong Alice: nếu `x + delta in bob_set` → trả `[x, x + delta]`.

Mảnh 1 + 2 đã có ở trên; ghép lại:

In [5]:
def fair_swap_set(alice: list[int], bob: list[int]) -> list[int]:
    delta = (sum(bob) - sum(alice)) // 2
    bob_set = set(bob)
    for x in alice:
        y = x + delta
        if y in bob_set:
            return [x, y]
    return []

print(fair_swap_set([1, 1], [2, 2]))
print(fair_swap_set([1, 2], [2, 3]))
print(fair_swap_set([2], [1, 3]))

[1, 2]
[1, 2]
[2, 3]


### 4.1. Ý nghĩa dấu của `delta`

- `delta > 0`: Bob nhiều kẹo hơn → Alice phải nhận hộp **to hơn** hộp mình đưa.
- `delta < 0`: Alice nhiều hơn → Alice nhận hộp **nhỏ hơn**.
- `delta = 0` không xảy ra (đề nói tổng khác nhau).

Ví dụ: Alice `[5, 3]` (8), Bob `[1, 3]` (4) → `delta = -2`: Alice đưa `x`, nhận `x - 2`. `x = 5` → `y = 3` ✔ (Bob có 3). Kiểm: Alice `8-5+3 = 6`, Bob `4-3+5 = 6`.

In [6]:
print(fair_swap_set([5, 3], [1, 3]))

[5, 3]


## 5. Cách 3 — Sắp xếp + hai con trỏ

Không dùng hash: sắp xếp cả hai mảng tăng dần. Đặt `i` ở Alice, `j` ở Bob. Ta cần `bob[j] - alice[i] == delta`.
- Nếu hiệu **nhỏ hơn** `delta` → cần `bob[j]` lớn hơn → `j += 1`.
- Nếu hiệu **lớn hơn** `delta` → cần `alice[i]` lớn hơn → `i += 1`.
- Bằng → tìm thấy.

`sorted(list)` trả list **mới** đã sắp xếp (không đổi list gốc), `O(n log n)`.

In [7]:
def fair_swap_two_pointers(alice: list[int], bob: list[int]) -> list[int]:
    delta = (sum(bob) - sum(alice)) // 2
    A, Bs = sorted(alice), sorted(bob)
    i = j = 0
    while i < len(A) and j < len(Bs):
        diff = Bs[j] - A[i]
        if diff == delta:
            return [A[i], Bs[j]]
        if diff < delta:
            j += 1
        else:
            i += 1
    return []

print(fair_swap_two_pointers([1, 1], [2, 2]))
print(fair_swap_two_pointers([1, 2], [2, 3]))
print(fair_swap_two_pointers([2], [1, 3]))
print(fair_swap_two_pointers([5, 3], [1, 3]))

[1, 2]
[1, 2]
[2, 3]
[3, 1]


**Vì sao di chuyển con trỏ như vậy không bỏ sót?** Nếu `Bs[j] - A[i] < delta` thì với `i` hiện tại, mọi `Bs[j'] ≤ Bs[j]` (bên trái `j`) càng nhỏ → hiệu càng nhỏ → không thể bằng `delta`; còn với các `i' > i`, `A[i'] ≥ A[i]` nên `Bs[j] - A[i'] ≤ Bs[j] - A[i] < delta` → `Bs[j]` vô dụng với mọi `i'` còn lại → bỏ `j` an toàn. Lập luận đối xứng khi hiệu lớn hơn.

## 6. Cách 4 — Sắp xếp Bob + tìm kiếm nhị phân (`bisect`)

`bisect.bisect_left(arr, v)` trả về **vị trí đầu tiên** trong mảng đã sắp xếp mà phần tử ≥ `v`, trong `O(log m)`. Nếu vị trí đó hợp lệ và `arr[pos] == v` → `v` có trong mảng.

In [8]:
from bisect import bisect_left

arr = [1, 3, 3, 7, 9]
for v in (3, 4, 10):
    pos = bisect_left(arr, v)
    found = pos < len(arr) and arr[pos] == v
    print(f"bisect_left({arr}, {v}) = {pos} -> có {v}? {found}")

bisect_left([1, 3, 3, 7, 9], 3) = 1 -> có 3? True
bisect_left([1, 3, 3, 7, 9], 4) = 3 -> có 4? False
bisect_left([1, 3, 3, 7, 9], 10) = 5 -> có 10? False


In [9]:
def fair_swap_binary_search(alice: list[int], bob: list[int]) -> list[int]:
    delta = (sum(bob) - sum(alice)) // 2
    Bs = sorted(bob)
    for x in alice:
        y = x + delta
        pos = bisect_left(Bs, y)
        if pos < len(Bs) and Bs[pos] == y:
            return [x, y]
    return []

print(fair_swap_binary_search([1, 1], [2, 2]), fair_swap_binary_search([2], [1, 3]))

[1, 2] [2, 3]


## 7. TRACE từng bước

### 7.1. Trace cách set với `alice = [1, 2, 5]`, `bob = [2, 4]`

`SA = 8`, `SB = 6`, `delta = -1` → Alice cần nhận hộp ít hơn hộp đưa đi 1 viên.

| x | y = x + delta | y trong {2, 4}? | Hành động |
|---:|---:|:---:|---|
| 1 | 0 | không | tiếp |
| 2 | 1 | không | tiếp |
| 5 | 4 | **có** | trả `[5, 4]` |

Kiểm: Alice `8 - 5 + 4 = 7`, Bob `6 - 4 + 5 = 7` ✔.

In [10]:
def trace_fair_swap_set(alice, bob):
    SA, SB = sum(alice), sum(bob)
    delta = (SB - SA) // 2
    bob_set = set(bob)
    print(f"SA={SA}, SB={SB}, delta={delta}, bob_set={sorted(bob_set)}")
    for x in alice:
        y = x + delta
        hit = y in bob_set
        print(f"  x={x}: cần y={y} -> {'CÓ' if hit else 'không có'}")
        if hit:
            print(f"  Sau đổi: Alice={SA - x + y}, Bob={SB - y + x}")
            return [x, y]
    return []

print(trace_fair_swap_set([1, 2, 5], [2, 4]))

SA=8, SB=6, delta=-1, bob_set=[2, 4]
  x=1: cần y=0 -> không có
  x=2: cần y=1 -> không có
  x=5: cần y=4 -> CÓ
  Sau đổi: Alice=7, Bob=7
[5, 4]


### 7.2. Trace hai con trỏ trên cùng input

In [11]:
def trace_two_pointers(alice, bob):
    delta = (sum(bob) - sum(alice)) // 2
    A, Bs = sorted(alice), sorted(bob)
    i = j = 0
    print(f"A={A}, B={Bs}, delta={delta}")
    while i < len(A) and j < len(Bs):
        diff = Bs[j] - A[i]
        if diff == delta:
            print(f"  i={i}, j={j}: B[j]-A[i] = {Bs[j]}-{A[i]} = {diff} == delta -> trả [{A[i]}, {Bs[j]}]")
            return [A[i], Bs[j]]
        move = "j += 1 (cần B lớn hơn)" if diff < delta else "i += 1 (cần A lớn hơn)"
        print(f"  i={i}, j={j}: B[j]-A[i] = {Bs[j]}-{A[i]} = {diff} vs delta {delta} -> {move}")
        if diff < delta:
            j += 1
        else:
            i += 1
    return []

print(trace_two_pointers([1, 2, 5], [2, 4]))

A=[1, 2, 5], B=[2, 4], delta=-1
  i=0, j=0: B[j]-A[i] = 2-1 = 1 vs delta -1 -> i += 1 (cần A lớn hơn)
  i=1, j=0: B[j]-A[i] = 2-2 = 0 vs delta -1 -> i += 1 (cần A lớn hơn)
  i=2, j=0: B[j]-A[i] = 2-5 = -3 vs delta -1 -> j += 1 (cần B lớn hơn)
  i=2, j=1: B[j]-A[i] = 4-5 = -1 == delta -> trả [5, 4]
[5, 4]


## 8. Code nộp LeetCode

In [12]:
class Solution:
    def fairCandySwap(self, aliceSizes: list[int], bobSizes: list[int]) -> list[int]:
        delta = (sum(bobSizes) - sum(aliceSizes)) // 2
        bob_set = set(bobSizes)
        for x in aliceSizes:
            if x + delta in bob_set:
                return [x, x + delta]
        return []          # không xảy ra vì đề đảm bảo có đáp án

sol = Solution()
print(sol.fairCandySwap([1, 1], [2, 2]), sol.fairCandySwap([1, 2], [2, 3]), sol.fairCandySwap([2], [1, 3]))

[1, 2] [1, 2] [2, 3]


### 8.1. Bản nộp hai con trỏ (không dùng hash)

In [13]:
class SolutionTwoPointers:
    def fairCandySwap(self, aliceSizes: list[int], bobSizes: list[int]) -> list[int]:
        delta = (sum(bobSizes) - sum(aliceSizes)) // 2
        A, Bs = sorted(aliceSizes), sorted(bobSizes)
        i = j = 0
        while i < len(A) and j < len(Bs):
            diff = Bs[j] - A[i]
            if diff == delta:
                return [A[i], Bs[j]]
            if diff < delta:
                j += 1
            else:
                i += 1
        return []

print(SolutionTwoPointers().fairCandySwap([1, 2], [2, 3]))

[1, 2]


## 9. Vì sao đúng?

**Tính đúng của điều kiện:** đã biến đổi tương đương: `SA - x + y = SB - y + x` ⇔ `y = x + delta`. Mọi cặp ta trả về đều thỏa → **không trả nhầm**.

**Không bỏ sót (cách set):** nếu tồn tại đáp án `(x*, y*)` thì `y* = x* + delta` và `y* ∈ bob_set`. Vòng lặp duyệt mọi `x` của Alice, nên khi tới `x*` (hoặc sớm hơn, gặp một đáp án khác) chắc chắn trả về.

**Invariant (cách set):** trước khi xét `x` thứ `k`, không có hộp nào trong `k` hộp đầu của Alice ghép được với Bob.

**Hai con trỏ:** đã lập luận ở mục 5 — mỗi bước loại bỏ một phần tử không thể nằm trong đáp án, nên nếu đáp án tồn tại, hai con trỏ không bao giờ "đi qua" nó.

## 10. Độ phức tạp và so sánh

`n = len(aliceSizes)`, `m = len(bobSizes)`.

| Cách | Ý tưởng | Thời gian | Bộ nhớ thêm | Ghi chú |
|---|---|---|---|---|
| 1. Brute force | thử mọi cặp | `O(n·m)` | `O(1)` | 10^8 phép khi max → TLE |
| 2. Phương trình + set | tra `x + delta` | `O(n + m)` trung bình | `O(m)` | **Khuyên dùng** |
| 3. Sort + hai con trỏ | quét hai mảng | `O(n log n + m log m)` | `O(n + m)` (bản sao sorted) | Không phụ thuộc hash |
| 4. Sort Bob + bisect | tìm nhị phân | `O(m log m + n log m)` | `O(m)` | Dùng thư viện `bisect` |

(Có thể dùng `list.sort()` tại chỗ để giảm bộ nhớ, nhưng sẽ làm thay đổi input.)

## 11. Bộ test

Đề cho phép **nhiều đáp án**, nên không so sánh với một output cố định. Ta viết hàm `is_valid(alice, bob, ans)` kiểm tra: `x` thuộc Alice, `y` thuộc Bob và sau khi đổi tổng bằng nhau.

In [14]:
def is_valid(alice, bob, ans) -> bool:
    if len(ans) != 2:
        return False
    x, y = ans
    return x in alice and y in bob and sum(alice) - x + y == sum(bob) - y + x

def check(name, alice, bob):
    for f in methods:
        ans = f(alice, bob)
        assert is_valid(alice, bob, ans), (name, f, ans)
    print(f"PASS  {name}: ví dụ đáp án {Solution().fairCandySwap(alice, bob)}")

methods = [fair_swap_bruteforce, fair_swap_set, fair_swap_two_pointers, fair_swap_binary_search,
           Solution().fairCandySwap, SolutionTwoPointers().fairCandySwap]

check("đề VD1", [1, 1], [2, 2])
check("đề VD2", [1, 2], [2, 3])
check("đề VD3", [2], [1, 3])
check("Alice nhiều hơn (delta âm)", [5, 3], [1, 3])
check("nhiều đáp án", [1, 2, 3, 4], [2, 3, 4, 5])
check("phần tử trùng", [4, 4, 4, 1], [3, 3, 3, 2])
check("Alice chỉ 1 hộp", [4], [1, 5])

PASS  đề VD1: ví dụ đáp án [1, 2]
PASS  đề VD2: ví dụ đáp án [1, 2]
PASS  đề VD3: ví dụ đáp án [2, 3]
PASS  Alice nhiều hơn (delta âm): ví dụ đáp án [5, 3]
PASS  nhiều đáp án: ví dụ đáp án [1, 3]
PASS  phần tử trùng: ví dụ đáp án [4, 3]
PASS  Alice chỉ 1 hộp: ví dụ đáp án [4, 5]


### 11.1. Kiểm thử ngẫu nhiên: sinh input **chắc chắn có đáp án**

Cách sinh: tạo hai mảng ngẫu nhiên, chọn `x` ở Alice và `y` ở Bob, rồi điều chỉnh để tổng sau đổi bằng nhau bằng cách thêm một hộp "cân bằng" có kích thước phù hợp. Đơn giản hơn: sinh ngẫu nhiên và **chỉ giữ** các input mà brute force tìm thấy đáp án và tổng khác nhau.

In [15]:
import random
random.seed(888)
tested = 0
while tested < 400:
    alice = [random.randint(1, 10) for _ in range(random.randint(1, 6))]
    bob = [random.randint(1, 10) for _ in range(random.randint(1, 6))]
    if sum(alice) == sum(bob) or not fair_swap_bruteforce(alice, bob):
        continue                      # không thỏa ràng buộc đề -> bỏ
    for f in methods:
        assert is_valid(alice, bob, f(alice, bob))
    tested += 1
print("400 input hợp lệ ngẫu nhiên: mọi cách đều trả đáp án hợp lệ.")

400 input hợp lệ ngẫu nhiên: mọi cách đều trả đáp án hợp lệ.


In [16]:
# Input lớn: n = m = 10^4 -> cách set chạy tức thì
import time
alice = list(range(1, 10_001))
bob = list(range(3, 10_003))           # mỗi hộp Bob = hộp Alice + 2  -> delta = 10^4
t0 = time.perf_counter()
ans = Solution().fairCandySwap(alice, bob)
print("Đáp án:", ans, "| hợp lệ:", is_valid(alice, bob, ans), f"| {time.perf_counter() - t0:.4f}s")

Đáp án: [1, 10001] | hợp lệ: True | 0.0006s


Ở ví dụ lớn: `SB - SA = 2 · 10^4` → `delta = 10^4`, nên cặp hợp lệ là `x` và `x + 10^4` (ví dụ `[1, 10001]`). Brute force với 10^8 cặp ở đây sẽ rất chậm nên ta không chạy.

## 12. Lỗi người mới hay gặp

### Lỗi 1: Quên chia 2 — dùng `delta = SB - SA`
Nhầm "chênh lệch tổng" với "chênh lệch hai hộp". Đổi hộp làm **một người tăng, người kia giảm** cùng lúc, nên hiệu hai hộp chỉ cần bằng **một nửa** chênh lệch.

In [17]:
def wrong_no_half(alice, bob):
    delta = sum(bob) - sum(alice)          # SAI: thiếu // 2
    s = set(bob)
    for x in alice:
        if x + delta in s:
            return [x, x + delta]
    return []

print("Sai :", wrong_no_half([1, 1], [2, 2]), "| Đúng:", fair_swap_set([1, 1], [2, 2]))

Sai : [] | Đúng: [1, 2]


### Lỗi 2: Đảo dấu — `delta = (SA - SB) // 2`
Khi đó ta tìm `y = x - đúng_delta` → trả cặp làm chênh lệch **tăng gấp đôi** hoặc không tìm thấy.

In [18]:
def wrong_sign(alice, bob):
    delta = (sum(alice) - sum(bob)) // 2   # SAI dấu
    s = set(bob)
    for x in alice:
        if x + delta in s:
            return [x, x + delta]
    return []

print("Sai :", wrong_sign([2], [1, 3]), "| Đúng:", fair_swap_set([2], [1, 3]))
print("Kiểm cặp sai [2,1]: Alice =", 2 - 2 + 1, ", Bob =", 4 - 1 + 2)

Sai : [2, 1] | Đúng: [2, 3]
Kiểm cặp sai [2,1]: Alice = 1 , Bob = 5


### Lỗi 3: Dùng `x + delta in bobSizes` (list) thay vì set
Đúng kết quả nhưng mỗi lần `in` trên list là `O(m)` → tổng `O(n·m)`, quay về brute force.

### Lỗi 4: Trả `[y, x]` (sai thứ tự)
Đề quy định `answer[0]` là hộp **Alice**, `answer[1]` là hộp **Bob**.

### Lỗi 5: Tính `sum()` bên trong vòng lặp
`for x in alice: if sum(bob) - sum(alice) ...` → mỗi vòng tốn `O(n + m)` → `O(n·(n+m))`. Tính tổng **một lần** trước vòng lặp.

## 13. Mẹo nhận dạng pattern

- "Đổi / chuyển một phần tử để hai bên **bằng nhau**" → viết phương trình tổng trước & sau, rút ra **hiệu cần tìm** (`delta`).
- Sau khi có "với mỗi `x`, cần tìm `y = f(x)`" → **hash set** (giống Two Sum với `needed = target - x`).
- Hai mảng + tìm cặp có **hiệu/tổng cố định** → nếu không muốn hash: **sắp xếp + hai con trỏ** hoặc **bisect**.
- Đề "trả bất kỳ đáp án" → test bằng **hàm kiểm tra tính hợp lệ**, không so với một output cố định.

Bài cùng họ: 1 Two Sum, 167 Two Sum II, 532 K-diff Pairs, 1865 Finding Pairs With a Certain Sum.

## 14. Biến thể

### 14.1. Trả về **tất cả** cặp giá trị khác nhau có thể đổi
### 14.2. Đếm số **cặp chỉ số** `(i, j)` hợp lệ (hộp trùng kích thước tính riêng)
`Counter` của Bob cho biết có bao nhiêu hộp mang giá trị `y`.

In [19]:
from collections import Counter

def all_fair_swaps(alice, bob):
    delta = (sum(bob) - sum(alice)) // 2
    bob_set = set(bob)
    return sorted({(x, x + delta) for x in alice if x + delta in bob_set})

def count_index_pairs(alice, bob):
    delta = (sum(bob) - sum(alice)) // 2
    cb = Counter(bob)
    return sum(cb[x + delta] for x in alice)

print("Mọi cặp giá trị:", all_fair_swaps([1, 2, 3, 4], [2, 3, 4, 5]))
print("Số cặp chỉ số  :", count_index_pairs([1, 1, 2], [2, 2, 3]))

Mọi cặp giá trị: [(1, 3), (2, 4), (3, 5)]
Số cặp chỉ số  : 5


Ở ví dụ thứ hai: `SA = 4`, `SB = 7` → hiệu 3 lẻ → **không** có đáp án (vi phạm đảm bảo của đề), và `// 2` làm tròn xuống `1` → đếm ra cặp **sai**! Đây là lời nhắc: khi bỏ đảm bảo của đề, phải kiểm `(SB - SA) % 2 == 0` trước.

In [20]:
def count_index_pairs_safe(alice, bob):
    diff = sum(bob) - sum(alice)
    if diff % 2 != 0:
        return 0
    delta = diff // 2
    cb = Counter(bob)
    return sum(cb[x + delta] for x in alice)

print("Bản an toàn:", count_index_pairs_safe([1, 1, 2], [2, 2, 3]))
print("Có đáp án  :", count_index_pairs_safe([1, 1, 2], [2, 2, 4]), "(SA=4, SB=8, delta=2: x=1->3? không; x=2->4 có 1)")

Bản an toàn: 0
Có đáp án  : 1 (SA=4, SB=8, delta=2: x=1->3? không; x=2->4 có 1)


## 15. Bài tập tự luyện

Hàm TODO chỉ định nghĩa, không được gọi khi chạy notebook.

### Bài 1 — Tính tay
`alice = [1, 2, 5]`, `bob = [2, 4]`: tính `SA`, `SB`, `delta` rồi tìm **mọi** cặp hợp lệ.

### Bài 2 — Dự đoán output
`bisect_left([2, 4, 4, 8], 4)` và `bisect_left([2, 4, 4, 8], 5)` trả gì?

### Bài 3 — Điền chỗ trống: cách set

In [21]:
def fair_swap_exercise(alice, bob):
    # TODO 1: delta = (sum(bob) - sum(alice)) // 2
    # TODO 2: tạo set cho bob
    # TODO 3: duyệt x, kiểm x + delta
    pass

# assert is_valid([1, 2], [2, 3], fair_swap_exercise([1, 2], [2, 3]))
# assert is_valid([2], [1, 3], fair_swap_exercise([2], [1, 3]))

### Bài 4 — Tự viết bản hai con trỏ

In [22]:
def fair_swap_tp_exercise(alice, bob):
    # TODO: sort hai mảng, i/j, so sánh hiệu với delta
    pass

# assert is_valid([5, 3], [1, 3], fair_swap_tp_exercise([5, 3], [1, 3]))

### Bài 5 — Biến thể: đổi để Alice có **nhiều hơn Bob đúng `k` kẹo**
Trả `[x, y]` bất kỳ hoặc `[]` nếu không có. Gợi ý: viết lại phương trình `SA - x + y = SB - y + x + k`.

In [23]:
def swap_to_lead_by_k(alice, bob, k):
    # TODO
    pass

# assert swap_to_lead_by_k([1, 5], [2, 2], 0) == []        # tổng 6 vs 4
# assert swap_to_lead_by_k([1, 5], [2, 2], 4) == [1, 2]   # Alice 7, Bob 3

### Bài 6 — Mini challenge: chọn cặp sao cho **hộp Alice đưa đi nhỏ nhất**
Trả về `[x, y]` hợp lệ với `x` nhỏ nhất có thể (nếu nhiều `y`, chỉ có một vì `y = x + delta`).

In [24]:
def fair_swap_min_x(alice, bob):
    # TODO
    pass

# assert fair_swap_min_x([1, 2, 3, 4], [2, 3, 4, 5]) == [1, 3]

## 16. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | tổng hai bên | delta âm | `SA=8, SB=6, delta=-1` → thử x=1,2,5 |
| 2 | vị trí đầu tiên ≥ v | 4 nằm ở chỉ số 1 | `1` và `3` |
| 3 | Two Sum | `bob_set = set(bob)` | `if x + delta in bob_set: return [x, x + delta]` |
| 4 | sorted | `diff < delta → j += 1` | ngược lại `i += 1` |
| 5 | phương trình mới | `2(y - x) = SB - SA + k` | kiểm chẵn lẻ rồi tra set |
| 6 | duyệt x tăng dần | `sorted(set(alice))` | trả ngay cặp đầu tiên tìm được |

## 17. Đáp án tham khảo

**Bài 1:** `SA = 8`, `SB = 6`, `delta = -1`. `x = 1 → 0` (không), `x = 2 → 1` (không), `x = 5 → 4` (có). Cặp duy nhất `[5, 4]`.

**Bài 2:** `1` và `3`.

**Bài 5:** `SA - x + y = SB - y + x + k` ⇔ `2(y - x) = SB - SA + k` ⇔ `y = x + (SB - SA + k) / 2` (cần chẵn).

In [25]:
def ans_fair_swap(alice, bob):
    delta = (sum(bob) - sum(alice)) // 2
    bob_set = set(bob)
    for x in alice:
        if x + delta in bob_set:
            return [x, x + delta]
    return []

def ans_fair_swap_tp(alice, bob):
    delta = (sum(bob) - sum(alice)) // 2
    A, Bs = sorted(alice), sorted(bob)
    i = j = 0
    while i < len(A) and j < len(Bs):
        d = Bs[j] - A[i]
        if d == delta:
            return [A[i], Bs[j]]
        if d < delta:
            j += 1
        else:
            i += 1
    return []

def ans_swap_to_lead_by_k(alice, bob, k):
    diff = sum(bob) - sum(alice) + k
    if diff % 2:
        return []
    delta = diff // 2
    bob_set = set(bob)
    for x in alice:
        if x + delta in bob_set:
            return [x, x + delta]
    return []

def ans_fair_swap_min_x(alice, bob):
    delta = (sum(bob) - sum(alice)) // 2
    bob_set = set(bob)
    for x in sorted(set(alice)):
        if x + delta in bob_set:
            return [x, x + delta]
    return []

assert all_fair_swaps([1, 2, 5], [2, 4]) == [(5, 4)]
assert bisect_left([2, 4, 4, 8], 4) == 1 and bisect_left([2, 4, 4, 8], 5) == 3
assert is_valid([1, 2], [2, 3], ans_fair_swap([1, 2], [2, 3]))
assert is_valid([2], [1, 3], ans_fair_swap([2], [1, 3]))
assert is_valid([5, 3], [1, 3], ans_fair_swap_tp([5, 3], [1, 3]))
assert ans_swap_to_lead_by_k([1, 5], [2, 2], 0) == []
x, y = ans_swap_to_lead_by_k([1, 5], [2, 2], 4)
assert [x, y] == [1, 2] and sum([1, 5]) - x + y == sum([2, 2]) - y + x + 4
assert ans_fair_swap_min_x([1, 2, 3, 4], [2, 3, 4, 5]) == [1, 3]
print("Đáp án tham khảo PASS toàn bộ test! (ví dụ bài 5:", [x, y], ")")

Đáp án tham khảo PASS toàn bộ test! (ví dụ bài 5: [1, 2] )


## 18. Template nhớ nhanh và checklist submit

```python
class Solution:
    def fairCandySwap(self, aliceSizes, bobSizes):
        delta = (sum(bobSizes) - sum(aliceSizes)) // 2   # y - x phải bằng delta
        bob_set = set(bobSizes)                          # tra cứu O(1) trung bình
        for x in aliceSizes:
            if x + delta in bob_set:
                return [x, x + delta]                    # [hộp Alice, hộp Bob]
```

**Công thức nhớ:** `SA - x + y = SB - y + x  ⇒  y = x + (SB - SA) / 2`.

**Checklist:**
- [ ] `delta` có **chia 2** và đúng dấu `(SB - SA)`?
- [ ] Tra cứu trên **set**, không phải list?
- [ ] Tính `sum` một lần, ngoài vòng lặp?
- [ ] Trả đúng thứ tự `[hộp Alice, hộp Bob]`?
- [ ] Test bằng hàm kiểm tra hợp lệ (vì có nhiều đáp án)?